[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArthurLimaS/ir-med/blob/main/example_notebook.ipynb)

# If on Google Colab

In [2]:
RunningInCOLAB = 'google.colab' in str(get_ipython()) if hasattr(__builtins__,'__IPYTHON__') else False

In [3]:
if RunningInCOLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

    !git clone https://github.com/ArthurLimaS/ir-med.git
    %cd ir-med
    !pip install -r "requirements.txt"

Cloning into 'ir-med'...
remote: Enumerating objects: 304, done.
remote: Counting objects: 100% (304/304), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 304 (delta 165), reused 235 (delta 115), pack-reused 0 (from 0)
Receiving objects: 100% (304/304), 29.90 MiB | 16.33 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/ir-med


# Loading CMED data

In [4]:
import etl_functions as etl

# Download nltk 'punkt_tab' data [REQUIRED FOR SOME OF THE IR-MED FUNCTIONS]
etl.download_nltk_punkt_tab()

# Load CMED data

df_cmed = etl.load_cmed(path = "data/cmed/cmed_clean_2024_03.csv")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
df_cmed

,SUBSTÂNCIA,CÓDIGO GGREM,REGISTRO,EAN 1,EAN 2,EAN 3,PRODUTO,APRESENTAÇÃO
0,21-ACETATO DE DEXAMETASONA;CLOTRIMAZOL,538912020009303,1705600230032,7891106000956,-,-,BAYCUTEN N,"10 MG/G + 0,443 MG/G CREM DERM CT BG AL X 40 G"
1,ABATACEPTE,505107701157215,1018003900019,7896016806469,-,-,ORENCIA,250 MG PO LIOF SOL INJ CT 1 FA + SER DESCARTÁVEL
2,ABATACEPTE,505113030019605,1018003900027,7896016807442,-,-,ORENCIA,125 MG/ML SOL INJ SC CT SER PREENCHIDA
3,ABATACEPTE,505113100020505,1018003900078,7896016808197,-,-,ORENCIA,125 MG/ML SOL INJ SC CT 4 SER PREENC VD TRANS ...
4,ABCIXIMABE,514517110034217,1123634150015,7896212452453,-,-,REOPRO,2 MG/ML SOL INJ CT FA VD INC X 5 ML
...,...,...,...,...,...,...,...,...
29861,ÓXIDO DE MAGNÉSIO;SIMETICONA;HIDRÓXIDO DE ALUM...,508011804138416,1004306960107,7891317469610,7891317020118,-,SIMECO PLUS,120 MG/ML + 60 MG/ML + 7 MG/ML SUS OR CT FR VD...
29862,ÓXIDO DE ZINCO,533507401164427,1039201400023,7898049793914,-,-,VITAGLÓS,5500UI/G + 990UI/G + 150MG/G POM CT BG AL X 45G
29863,ÓXIDO DE ZINCO,533507402160425,1039201400031,7898049791309,-,-,VITAGLÓS,5500UI/G + 990UI/G + 150MG/G POM CX 50 BG AL X...
29864,ÓXIDO DE ZINCO;RETINOL;COLECALCIFEROL,528526201160419,1256801730019,7898148295500,-,-,PRATIGLÓS,5000 UI/G + 900 UI/G + 150 MG/G POM CX 50 BG A...


### Adjust column names

In [6]:
df_cmed = etl.std_cols_names_preprocessing(df_cmed)

In [7]:
df_cmed

,substancia,codigo_ggrem,registro,ean_1,ean_2,ean_3,produto,apresentacao
0,21-ACETATO DE DEXAMETASONA;CLOTRIMAZOL,538912020009303,1705600230032,7891106000956,-,-,BAYCUTEN N,"10 MG/G + 0,443 MG/G CREM DERM CT BG AL X 40 G"
1,ABATACEPTE,505107701157215,1018003900019,7896016806469,-,-,ORENCIA,250 MG PO LIOF SOL INJ CT 1 FA + SER DESCARTÁVEL
2,ABATACEPTE,505113030019605,1018003900027,7896016807442,-,-,ORENCIA,125 MG/ML SOL INJ SC CT SER PREENCHIDA
3,ABATACEPTE,505113100020505,1018003900078,7896016808197,-,-,ORENCIA,125 MG/ML SOL INJ SC CT 4 SER PREENC VD TRANS ...
4,ABCIXIMABE,514517110034217,1123634150015,7896212452453,-,-,REOPRO,2 MG/ML SOL INJ CT FA VD INC X 5 ML
...,...,...,...,...,...,...,...,...
29861,ÓXIDO DE MAGNÉSIO;SIMETICONA;HIDRÓXIDO DE ALUM...,508011804138416,1004306960107,7891317469610,7891317020118,-,SIMECO PLUS,120 MG/ML + 60 MG/ML + 7 MG/ML SUS OR CT FR VD...
29862,ÓXIDO DE ZINCO,533507401164427,1039201400023,7898049793914,-,-,VITAGLÓS,5500UI/G + 990UI/G + 150MG/G POM CT BG AL X 45G
29863,ÓXIDO DE ZINCO,533507402160425,1039201400031,7898049791309,-,-,VITAGLÓS,5500UI/G + 990UI/G + 150MG/G POM CX 50 BG AL X...
29864,ÓXIDO DE ZINCO;RETINOL;COLECALCIFEROL,528526201160419,1256801730019,7898148295500,-,-,PRATIGLÓS,5000 UI/G + 900 UI/G + 150 MG/G POM CX 50 BG A...


# Modeling Phase

### Words pre-processing

In [8]:
from tqdm.notebook import tqdm

for idx, row in tqdm(df_cmed.iterrows(), desc = 'CMED words preprocessing',
                     total = df_cmed.shape[0]):
    df_cmed.at[idx, 'substancia'] = etl.std_preprocessing(row['substancia'],
                                                          rem_nums = True,
                                                          rem_stopwords_ai = True,
                                                          correct_ai = True,
                                                          rem_rep_tokens = True)

    df_cmed.at[idx, 'apresentacao'] = etl.std_preprocessing(row['apresentacao'],
                                                            rem_stopwords_pr = True)

CMED words preprocessing:   0%|          | 0/29866 [00:00<?, ?it/s]

In [9]:
df_cmed

,substancia,codigo_ggrem,registro,ean_1,ean_2,ean_3,produto,apresentacao
0,dexametasona clotrimazol,538912020009303,1705600230032,7891106000956,-,-,BAYCUTEN N,10 mg g 0 443 mg g crem derm ct bg al x 40 g
1,abatacepte,505107701157215,1018003900019,7896016806469,-,-,ORENCIA,250 mg po liof sol inj ct 1 fa ser descartavel
2,abatacepte,505113030019605,1018003900027,7896016807442,-,-,ORENCIA,125 mg ml sol inj sc ct ser preenc
3,abatacepte,505113100020505,1018003900078,7896016808197,-,-,ORENCIA,125 mg ml sol inj sc ct 4 ser preenc vd trans ...
4,abciximabe,514517110034217,1123634150015,7896212452453,-,-,REOPRO,2 mg ml sol inj ct fa vd inc x 5 ml
...,...,...,...,...,...,...,...,...
29861,magnesio simeticona al,508011804138416,1004306960107,7891317469610,7891317020118,-,SIMECO PLUS,120 mg ml 60 mg ml 7 mg ml sus or ct fr vd amb...
29862,zinco,533507401164427,1039201400023,7898049793914,-,-,VITAGLÓS,5500 ui g 990 ui g 150 mg g pom ct bg al x 45 g
29863,zinco,533507402160425,1039201400031,7898049791309,-,-,VITAGLÓS,5500 ui g 990 ui g 150 mg g pom cx 50 bg al x ...
29864,zinco retinol colecalciferol,528526201160419,1256801730019,7898148295500,-,-,PRATIGLÓS,5000 ui g 900 ui g 150 mg g pom cx 50 bg al x ...


### Clustering CMED items by their Active Ingredients

In [10]:
grouped_cmed = etl.group_cmed(df_cmed, 'substancia', verbose = True)

Grouping active ingredients:   0%|          | 0/2140 [00:00<?, ?it/s]

In [11]:
grouped_cmed

,key,key_sorted,indexes
0,abacavir,abacavir,"[27317, 27318]"
1,abatacepte,abatacepte,"[1, 2, 3]"
2,abciximabe,abciximabe,"[4, 5]"
3,abemaciclibe,abemaciclibe,"[6, 7, 8, 9, 10, 11, 12, 13]"
4,abiraterona,abiraterona,"[121, 122, 123, 124, 125, 126, 127, 128, 129, ..."
...,...,...,...
2067,zoledronico,zoledronico,"[29700, 29701, 29702, 29703, 29704, 29705, 297..."
2068,zolmitriptana,zolmitriptana,"[29378, 29379]"
2069,zolpidem,zolpidem,"[16912, 16913, 16914, 16915, 16916, 16917, 169..."
2070,zopiclona,zopiclona,[29380]


### Identifying relevant words

In [12]:
import ir_med

answer_dict = ir_med.identify_relevant_words(df_cmed,
                                             columns=['substancia', 'apresentacao'])

In [13]:
cmed_ai_words = answer_dict['substancia']
cmed_ai_words

array(['abacavir', 'abatacepte', 'abciximabe', ..., 'zopiclona', 'zoster',
       'zuclopentixol'], dtype='<U30')

In [14]:
cmed_pr_words = answer_dict['apresentacao']
cmed_pr_words

array(['0', '00', '000', ..., 'xamp', 'xpe', 'xpect'], dtype='<U16')

# Load the list of medicines extracted from a public notice

In [15]:
df_notice = etl.load_notice('data/notices/csvs/LE_1616_2023_14.csv', sep = ';')

In [16]:
df_notice

,catmat,desc,und,quant,valor_unit,valor_total
0,271689,ÁCIDO \nASCÓRBICO \nconcentração/dosagem \n200...,Frasco 20 mL,198500,"R$ 1,34",265990.00
1,278489,"ÁCIDO FÓLICO concentração/dosagem \n0,2 mg/m...",FRASCO 30 mL,213400,"R$ 4,30",917620.00
2,315056,ÁGUA PARA INJEÇÃO,AMPOLA 10 mL,2311400,"R$ 0,28",647192.00
3,397502,"AGULHA \nHIPODÉRMICA, \nMATERIAL:AÇO \nINOXIDÁ...",CAIXA COM 100 \nUNIDADES,135900,"R$ 9,44",1282896.00
4,269941,"ÁLCOOL \nETÍLICO, \nHIDRATADO, \n70%(70°GL), L...",FRASCO COM \n1000 ML,200654,"R$ 5,39",1081525.06
...,...,...,...,...,...,...
72,292344,SULFATO \nFERROSO \nconcentração/dosagem 40 ...,COMPRIMIDO,6586400,"R$ 0,03",197592.00
73,332468,SULFATO \nFERROSO \nconcentração/dosagem 5 mg/...,FRASCO 100 mL,86420,"R$ 2,68",231605.60
74,272581,TIMOLOL \n– \nMALEATO \nconcentração/dosagem ...,FRASCO 5 mL,23610,"R$ 2,83",66816.30
75,328529,VALPROATO DE SÓDIO \nou ÁCIDO \nVALPRÓICO \...,CÁPSULA OU \nCOMPRIMIDO,2472100,"R$ 0,32",791072.00


### Drop unrelevant columns

In [17]:
df_notice = df_notice[['desc', 'und']]

In [18]:
df_notice

,desc,und
0,ÁCIDO \nASCÓRBICO \nconcentração/dosagem \n200...,Frasco 20 mL
1,"ÁCIDO FÓLICO concentração/dosagem \n0,2 mg/m...",FRASCO 30 mL
2,ÁGUA PARA INJEÇÃO,AMPOLA 10 mL
3,"AGULHA \nHIPODÉRMICA, \nMATERIAL:AÇO \nINOXIDÁ...",CAIXA COM 100 \nUNIDADES
4,"ÁLCOOL \nETÍLICO, \nHIDRATADO, \n70%(70°GL), L...",FRASCO COM \n1000 ML
...,...,...
72,SULFATO \nFERROSO \nconcentração/dosagem 40 ...,COMPRIMIDO
73,SULFATO \nFERROSO \nconcentração/dosagem 5 mg/...,FRASCO 100 mL
74,TIMOLOL \n– \nMALEATO \nconcentração/dosagem ...,FRASCO 5 mL
75,VALPROATO DE SÓDIO \nou ÁCIDO \nVALPRÓICO \...,CÁPSULA OU \nCOMPRIMIDO


# Ir-med execution

### Preprocessing of Public Notice Data

In [19]:
import re

# Create copy of original descriptions
df_notice['original_desc'] = df_notice['desc']

# Creation of the column where the indices of the CMED will be stored
df_notice['cmed_indexes'] = ""

# Apply the preprocess function to the columns 'descrição' and 'unidade'
for idx, row in tqdm(df_notice.iterrows(), desc = "Notice words preprocessing",
                     total = df_notice.shape[0]):
    # Remove linebreak in the original description
    df_notice.at[idx, 'original_desc'] = re.sub('\n', '', row['original_desc'])

    df_notice.at[idx, 'desc'] = etl.std_preprocessing(row['desc'],
                                                      correct_ai = True,
                                                      rem_rep_tokens = True)

    df_notice.at[idx, 'und'] = etl.std_preprocessing(row['und'],
                                                     rem_stopwords_pr = True)

/tmp/ipython-input-1610188768.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_notice['original_desc'] = df_notice['desc']
/tmp/ipython-input-1610188768.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_notice['cmed_indexes'] = ""


Notice words preprocessing:   0%|          | 0/77 [00:00<?, ?it/s]

In [20]:
df_notice

,desc,und,original_desc,cmed_indexes
0,acido ascorbico concentracao dosagem 200 mg ml...,fr 20 ml,ÁCIDO ASCÓRBICO concentração/dosagem 200 mg/mL...,
1,acido folico concentracao dosagem 0 2 mg ml fo...,fr 30 ml,"ÁCIDO FÓLICO concentração/dosagem 0,2 mg/mL,...",
2,agua para injecao,amp 10 ml,ÁGUA PARA INJEÇÃO,
3,agu hipodermica material aco inoxidavel silico...,cx com 100 unidades,"AGULHA HIPODÉRMICA, MATERIAL:AÇO INOXIDÁVEL SI...",
4,alcool etilico hidratado 70 deggl liq,fr com 1000 ml,"ÁLCOOL ETÍLICO, HIDRATADO, 70%(70°GL), LÍQUIDO",
...,...,...,...,...
72,sulfato ferroso concentracao dosagem 40 mg de ...,com,SULFATO FERROSO concentração/dosagem 40 mg ...,
73,sulfato ferroso concentracao dosagem 5 mg ml f...,fr 100 ml,"SULFATO FERROSO concentração/dosagem 5 mg/mL, ...",
74,timolol maleato concentracao dosagem 5 mg ml f...,fr 5 ml,TIMOLOL – MALEATO concentração/dosagem 5mg/mL...,
75,valproato de sodio ou acido valproico concentr...,cap ou com,VALPROATO DE SÓDIO ou ÁCIDO VALPRÓICO conce...,


### Information Retrival

In [21]:
SAVE_PROCESS_METADATA = True

for idx_X, row_X in tqdm(df_notice.iterrows(), desc= 'IR-Med prediction',
                         total = df_notice.shape[0]):

    desc_ai, desc_pr = ir_med.split_description(row_X['desc'], cmed_ai_words,
                                                cmed_pr_words)

    # Prediction using the IR-Med model
    df_notice.at[idx_X, 'cmed_indexes'], process_metadata = ir_med.predict(df_cmed,
                                                                           grouped_cmed,
                                                                           desc_ai,
                                                                           desc_pr,
                                                                           row_X['und'])

    if SAVE_PROCESS_METADATA:
        for key, value in process_metadata.items():
            df_notice.at[idx_X, key] = value

IR-Med prediction:   0%|          | 0/77 [00:00<?, ?it/s]

/tmp/ipython-input-56406694.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_notice.at[idx_X, key] = value


In [22]:
df_notice

,desc,und,original_desc,cmed_indexes,active_ingredient_found,desc_ai,similarity_value,desc_pr,quant_presentations_matched,size_cmed_filtered,pct_set_reduction
0,acido ascorbico concentracao dosagem 200 mg ml...,fr 20 ml,ÁCIDO ASCÓRBICO concentração/dosagem 200 mg/mL...,"[29442, 29444, 29452, 29462, 29468, 29471, 294...",ascorbico,ascorbico,1.000000,200 mg ml sol or got,8.0,63.0,0.873016
1,acido folico concentracao dosagem 0 2 mg ml fo...,fr 30 ml,"ÁCIDO FÓLICO concentração/dosagem 0,2 mg/mL,...","[29550, 29553]",folico,folico,1.000000,0 2 mg ml sol or,2.0,33.0,0.939394
2,agua para injecao,amp 10 ml,ÁGUA PARA INJEÇÃO,"[29752, 29826]",agua injecao,agua injecao,1.000000,injecao,2.0,39.0,0.948718
3,agu hipodermica material aco inoxidavel silico...,cx com 100 unidades,"AGULHA HIPODÉRMICA, MATERIAL:AÇO INOXIDÁVEL SI...","[1586, 1591, 1596, 1601, 1624, 1628]",apixabana,x,0.703704,agu hipodermica aco 21 g x 1 tipo conector em ...,6.0,94.0,0.936170
4,alcool etilico hidratado 70 deggl liq,fr com 1000 ml,"ÁLCOOL ETÍLICO, HIDRATADO, 70%(70°GL), LÍQUIDO",[29831],alcool polivinilico fenilefrina,alcool,0.838710,alcool 70 liq,1.0,1.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
72,sulfato ferroso concentracao dosagem 40 mg de ...,com,SULFATO FERROSO concentração/dosagem 40 mg ...,"[27751, 27753, 27754, 27757, 27758, 27767, 27768]",ferroso,ferroso ferro ii,0.858929,40 mg ii com or,7.0,23.0,0.695652
73,sulfato ferroso concentracao dosagem 5 mg ml f...,fr 100 ml,"SULFATO FERROSO concentração/dosagem 5 mg/mL, ...","[27744, 27749, 27762, 27763, 27765, 27766]",ferroso,ferroso,1.000000,5 mg ml xpe or,6.0,23.0,0.739130
74,timolol maleato concentracao dosagem 5 mg ml f...,fr 5 ml,TIMOLOL – MALEATO concentração/dosagem 5mg/mL...,"[20365, 20420, 20421, 20422, 20423, 20424, 204...",timolol,timolol,1.000000,5 mg ml,14.0,25.0,0.440000
75,valproato de sodio ou acido valproico concentr...,cap ou com,VALPROATO DE SÓDIO ou ÁCIDO VALPRÓICO conce...,"[29698, 29699]",valproico sodio,sodio valproico,1.000000,ou mg a 250 cap com or,2.0,8.0,0.750000


# Analysis of the Results

### Predict Active Ingredient Analysis

In [23]:
df_notice['got_ai_right'] = df_notice\
                            .apply(lambda row: ir_med.check_ai_prediction(row['desc'],
                                                                          row['desc_ai'],
                                                                          row['active_ingredient_found']),
                                    axis=1)

In [24]:
GOT_AI_RIGHT = False

df_notice[df_notice['got_ai_right'] == GOT_AI_RIGHT][['got_ai_right', 'desc', 'desc_ai', 'active_ingredient_found']]

,got_ai_right,desc,desc_ai,active_ingredient_found
3,False,agu hipodermica material aco inoxidavel silico...,x,apixabana
4,False,alcool etilico hidratado 70 deggl liq,alcool,alcool polivinilico fenilefrina
5,False,alcool etilico hidratado 70 gl gel,alcool,alcool polivinilico fenilefrina
17,False,cateter central aplicacao venoso materia prima...,,abacavir
18,False,cateter periferico aplicacao venoso modelo tip...,,abacavir
19,False,cateter periferico polimero radiopaco venoso a...,,abacavir
32,False,coletor de urina material plas tipo sist abert...,,abacavir
41,False,extrato medicinal principio ativo aquoso de ar...,extrato,extrato seco passiflora
64,False,polimixina b composicao associada com neomicin...,polimixina b neomicina fluocinolona lidocaina,clioquinol prednisolona polimixina b benzocaina
72,False,sulfato ferroso concentracao dosagem 40 mg de ...,ferroso ferro ii,ferroso


### Presentations Matched Analysis

In [25]:
df_notice['prs_matched'] = df_notice['cmed_indexes'] \
                           .apply(lambda x: etl.get_presentations(df_cmed, x))

df_notice['common_tokens_prs'] = df_notice['prs_matched']\
                                 .apply(lambda x: ir_med.get_common_tokens(x))

df_notice['got_pr_right'] = df_notice\
                            .apply(lambda row: ir_med.check_pr_predictions(row['desc_pr'],
                                                                           row['und'],
                                                                           row['common_tokens_prs']),
                                    axis=1)

In [26]:
GOT_PR_RIGHT = False

df_notice[df_notice['got_pr_right'] == GOT_AI_RIGHT][['got_pr_right', 'desc',
                                                      'und', 'desc_pr',
                                                      'common_tokens_prs']]

,got_pr_right,desc,und,desc_pr,common_tokens_prs
2,False,agua para injecao,amp 10 ml,injecao,"{ml, x, cx, amp, 10, sol, inj}"
3,False,agu hipodermica material aco inoxidavel silico...,cx com 100 unidades,agu hipodermica aco 21 g x 1 tipo conector em ...,"{plas, bl, 5, x, pvc, mg, 100, rev, ct, al, co..."
4,False,alcool etilico hidratado 70 deggl liq,fr com 1000 ml,alcool 70 liq,"{ml, 2, oft, plas, x, mg, 1, opc, ct, fr, 15, ..."
5,False,alcool etilico hidratado 70 gl gel,fr com 500 ml,alcool 70 gl gel,"{ml, 2, oft, plas, x, mg, 1, opc, ct, fr, 15, ..."
6,False,alopurinol concentracao dosagem 300 mg forma f...,com,300 mg com or,"{plas, bl, x, mg, ct, al, 300, com, trans}"
7,False,ambroxol cloridrato concentracao dosagem 3 mg ...,fr 100 ml,3 mg ml xpe or,"{3, ml, x, mg, 100, xpe, fr, amb}"
8,False,ambroxol cloridrato concentracao dosagem 6 mg ...,fr 100 ml,6 mg ml xpe or,"{ml, x, mg, 100, xpe, fr, 6}"
9,False,beclometasona dipropionato concentracao dosage...,fr 200 doses,50 mcg dose aer ou spr inal nas,"{sus, plas, x, 200, doses, mcg, ct, fr, valv, ..."
10,False,beclometasona dipropionato concentracao dosage...,fr 200 doses,200 mcg dose po inal or,"{inal, po, 200, x, 100, doses, mcg, ct, dose, ..."
11,False,beclometasona dipropionato concentracao dosage...,fr 200 doses,250 mcg dose aer ou spr inal nas,"{aer, x, 200, doses, mcg, ct, al, tb, sol, dos..."


# .csv export

In [27]:
df_notice

,desc,und,original_desc,cmed_indexes,active_ingredient_found,desc_ai,similarity_value,desc_pr,quant_presentations_matched,size_cmed_filtered,pct_set_reduction,got_ai_right,prs_matched,common_tokens_prs,got_pr_right
0,acido ascorbico concentracao dosagem 200 mg ml...,fr 20 ml,ÁCIDO ASCÓRBICO concentração/dosagem 200 mg/mL...,"[29442, 29444, 29452, 29462, 29468, 29471, 294...",ascorbico,ascorbico,1.000000,200 mg ml sol or got,8.0,63.0,0.873016,True,"[200 mg ml sol or ct fr plas got x 20 ml, 200 ...","{ml, x, 200, mg, 20, fr, got, or, sol}",True
1,acido folico concentracao dosagem 0 2 mg ml fo...,fr 30 ml,"ÁCIDO FÓLICO concentração/dosagem 0,2 mg/mL,...","[29550, 29553]",folico,folico,1.000000,0 2 mg ml sol or,2.0,33.0,0.939394,True,"[0 2 mg ml sol or ct fr got plas amb x 30 ml, ...","{ml, 2, plas, x, mg, 30, fr, or, sol, amb, 0}",True
2,agua para injecao,amp 10 ml,ÁGUA PARA INJEÇÃO,"[29752, 29826]",agua injecao,agua injecao,1.000000,injecao,2.0,39.0,0.948718,True,"[sol inj cx 100 amp poliet x 10 ml, sol inj cx...","{ml, x, cx, amp, 10, sol, inj}",False
3,agu hipodermica material aco inoxidavel silico...,cx com 100 unidades,"AGULHA HIPODÉRMICA, MATERIAL:AÇO INOXIDÁVEL SI...","[1586, 1591, 1596, 1601, 1624, 1628]",apixabana,x,0.703704,agu hipodermica aco 21 g x 1 tipo conector em ...,6.0,94.0,0.936170,False,[2 5 mg com rev ct bl al plas pvc pvdc trans x...,"{plas, bl, 5, x, pvc, mg, 100, rev, ct, al, co...",False
4,alcool etilico hidratado 70 deggl liq,fr com 1000 ml,"ÁLCOOL ETÍLICO, HIDRATADO, 70%(70°GL), LÍQUIDO",[29831],alcool polivinilico fenilefrina,alcool,0.838710,alcool 70 liq,1.0,1.0,0.000000,False,[1 2 mg ml 14 0 mg ml sol oft ct fr plas opc g...,"{ml, 2, oft, plas, x, mg, 1, opc, ct, fr, 15, ...",False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72,sulfato ferroso concentracao dosagem 40 mg de ...,com,SULFATO FERROSO concentração/dosagem 40 mg ...,"[27751, 27753, 27754, 27757, 27758, 27767, 27768]",ferroso,ferroso ferro ii,0.858929,40 mg ii com or,7.0,23.0,0.695652,False,"[40 mg com rev ct bl al plas pvc trans x 50, 4...","{plas, bl, x, mg, rev, 40, ct, al, com}",False
73,sulfato ferroso concentracao dosagem 5 mg ml f...,fr 100 ml,"SULFATO FERROSO concentração/dosagem 5 mg/mL, ...","[27744, 27749, 27762, 27763, 27765, 27766]",ferroso,ferroso,1.000000,5 mg ml xpe or,6.0,23.0,0.739130,True,"[10 mg ml xpe ct fr vd amb x 100 ml, 25 mg ml ...","{ml, x, mg, 100, xpe, fr, amb}",False
74,timolol maleato concentracao dosagem 5 mg ml f...,fr 5 ml,TIMOLOL – MALEATO concentração/dosagem 5mg/mL...,"[20365, 20420, 20421, 20422, 20423, 20424, 204...",timolol,timolol,1.000000,5 mg ml,14.0,25.0,0.440000,True,"[2 5 mg ml sol oft ct fr got plas opc x 5 ml, ...","{ml, plas, oft, 5, x, mg, fr, got}",True
75,valproato de sodio ou acido valproico concentr...,cap ou com,VALPROATO DE SÓDIO ou ÁCIDO VALPRÓICO conce...,"[29698, 29699]",valproico sodio,sodio valproico,1.000000,ou mg a 250 cap com or,2.0,8.0,0.750000,True,"[250 mg cap mole ct fr vd amb x 25, 250 mg cap...","{x, mg, 250, ct, fr, mole, cap, vd, amb}",False


In [28]:
df_notice.to_csv('algorithm_results.csv',
                 sep=';', decimal=',', index=False)